# Capacitação Ciência de Dados — Semana 5
## Regressão Linear, Regressão Logística e Séries Temporais

**Dataset:** Superstore Sales (Kaggle)  
**Período:** 2015 – 2018  
**Registros:** ~10.000 vendas de uma loja de varejo americana

In [ ]:
# Instala o Prophet caso ainda não esteja disponível no ambiente
# As demais bibliotecas já fazem parte da distribuição padrão do Anaconda/pip
# !pip install prophet --quiet

## 1. Introdução

### Sobre o dataset

O **Superstore Sales** é um dataset público disponível no Kaggle que simula as operações de uma loja de varejo americana entre 2015 e 2018. Ele contém informações sobre pedidos, clientes, produtos, descontos, vendas e lucros.

### Problemas propostos

| Modelo | Pergunta de negócio | Variável alvo |
|--------|--------------------|--------------|
| Regressão Linear | Quanto uma venda vai gerar em receita? | `Sales` (contínua) |
| Regressão Logística | Essa venda vai dar lucro ou prejuízo? | `Profit_flag` (binária) |
| Séries Temporais | Qual será a receita diária nos próximos 3 meses? | `Sales` agregado por dia |

### Justificativa das escolhas

- **Regressão linear:** `Sales` é uma variável contínua e positiva, influenciada por desconto, quantidade e categoria — relação razoavelmente linear após transformações.
- **Regressão logística:** Binarizar o lucro (`Profit > 0`) cria um problema de classificação relevante para o negócio: identificar vendas com risco de prejuízo antes de fechá-las.
- **Prophet:** O dataset cobre 4 anos de vendas diárias com sazonalidade semanal e anual visível, cenário ideal para o Prophet da Meta.

In [ ]:
# --- Imports gerais ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual consistente em todo o notebook
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 13

print('Bibliotecas carregadas com sucesso.')

In [ ]:
# Carregameto do dataset — ajuste o caminho se necessário
# Arquivo original do Kaggle: 'Sample - Superstore.csv'
df = pd.read_csv('Sample - Superstore.csv', encoding='latin-1')

print(f'Shape: {df.shape}')
df.head()

---
## 2. Análise Exploratória (EDA)

In [ ]:
# Visão geral dos tipos e valores ausentes
print('=== Tipos de dados ===')
print(df.dtypes)
print('\n=== Valores ausentes ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df[['Sales', 'Profit', 'Discount', 'Quantity']].describe().round(2)

In [ ]:
# Conversão da coluna de data — necessária para a série temporal
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=False)

print(f'Período: {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Total de pedidos únicos: {df["Order ID"].nunique()}')

In [ ]:
# --- Distribuição de Sales (faturamento bruto) e Profit (distribuição de lucros) ---
fig, axes = plt.subplots(1, 2)

# Aplicamos log1p para reduzir o efeito dos outliers na visualização
axes[0].hist(np.log1p(df['Sales']), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição de Sales (log1p)')
axes[0].set_xlabel('log(1 + Sales)')

axes[1].hist(df['Profit'], bins=40, color='coral', edgecolor='white')
axes[1].set_title('Distribuição de Profit')
axes[1].set_xlabel('Profit (USD)')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1, label='zero')
axes[1].legend()

plt.suptitle('Distribuições das variáveis alvo', y=1.02)
plt.tight_layout()
plt.show()

# Observação: Sales tem distribuição muito assimétrica — usaremos log1p como feature. >> para fins estabilizadores da variância e evitar divisões incorretas
# Profit apresenta vendas com prejuízo (< 0), o que motiva a criação do Profit_flag.

In [ ]:
# --- Relação entre desconto e lucro ---
plt.figure()
sns.scatterplot(
    data=df, x='Discount', y='Profit',
    hue='Category', alpha=0.4, s=20
)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('Desconto vs Lucro por categoria')
plt.xlabel('Discount')
plt.ylabel('Profit (USD)')
plt.tight_layout()
plt.show()

# Observação: descontos acima de ~0.4 estão fortemente associados a prejuízo,
# o que torna Discount um preditor importante para a regressão logística.

In [ ]:
# --- Matriz de correlação (variáveis numéricas) ---> resumo das relações entre variáveis
corr = df[['Sales', 'Profit', 'Discount', 'Quantity']].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Matriz de correlação')
plt.tight_layout()
plt.show()

In [ ]:
# --- Detecção de outliers em Sales (IQR) ---> voulme de transações que fogem do padrão 
Q1, Q3 = df['Sales'].quantile(0.25), df['Sales'].quantile(0.75)
IQR = Q3 - Q1
outlier_mask = (df['Sales'] < Q1 - 1.5 * IQR) | (df['Sales'] > Q3 + 1.5 * IQR)

print(f'Outliers em Sales: {outlier_mask.sum()} registros ({outlier_mask.mean():.1%} do total)')

# Estratégia: manteremos os outliers no conjunto completo pois são vendas legítimas
# (ex: equipamentos de escritório de alto valor). Na regressão usaremos log1p(Sales)
# para reduzir o impacto desses valores extremos.

### Justificativa das variáveis preditoras

| Feature | Tipo | Justificativa |
|---------|------|---------------|
| `Discount` | Numérica | Alta correlação negativa com Profit; impacta Sales indiretamente |
| `Quantity` | Numérica | Volume vendido influencia receita diretamente |
| `Category` | Categórica (OHE) | Três categorias com perfis de margem distintos |
| `Region` | Categórica (OHE) | Regiões com pricing e demanda diferente |
| `Segment` | Categórica (OHE) | Consumer, Corporate, Home Office têm tickets médios diferentes |

---
## 3. Regressão Linear

**Objetivo:** prever o valor de `Sales` de uma venda a partir de features operacionais.  
Utilizamos `log1p(Sales)` como alvo para lidar com a assimetria da distribuição.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# --- Preparação dos dados ---
# One-hot encoding nas variáveis categóricas
features_cat = ['Category', 'Region', 'Segment']
features_num = ['Discount', 'Quantity']

X_cat = pd.get_dummies(df[features_cat], drop_first=True)
X_num = df[features_num].copy()
X = pd.concat([X_num, X_cat], axis=1)

# Alvo com transformação log1p
y = np.log1p(df['Sales'])

# Divisão treino/teste (80/20) com seed fixo para reprodutibilidade >> 80% para treinamento e o restante em avaliação de desempenho
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Padronização das features numéricas >> pré processamento dos dados para que fiquem em mesma escala
scaler = StandardScaler()
X_train[features_num] = scaler.fit_transform(X_train[features_num])
X_test[features_num]  = scaler.transform(X_test[features_num])

print(f'Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')

In [ ]:
# --- Treinamento ---
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_log = lr_model.predict(X_test)

# Avaliação no espaço log
r2   = r2_score(y_test, y_pred_log)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_log))
mae  = mean_absolute_error(y_test, y_pred_log)

print('=== Métricas — espaço log1p(Sales) ===')
print(f'R²   : {r2:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')

In [ ]:
# --- Gráfico: real vs previsto (espaço original) ---
y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred_log)

fig, axes = plt.subplots(1, 2)

# Dispersão real vs previsto
axes[0].scatter(y_test_orig, y_pred_orig, alpha=0.3, s=15, color='steelblue')
lim = max(y_test_orig.max(), y_pred_orig.max())
axes[0].plot([0, lim], [0, lim], 'r--', linewidth=1, label='linha ideal')
axes[0].set_title('Real vs Previsto (Sales em USD)')
axes[0].set_xlabel('Sales real')
axes[0].set_ylabel('Sales previsto')
axes[0].legend()

# Distribuição dos resíduos (espaço log)
residuals = y_test.values - y_pred_log
axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_title('Distribuição dos resíduos (log)')
axes[1].set_xlabel('Resíduo')

plt.suptitle(f'Regressão Linear — R² = {r2:.3f}', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Importância dos coeficientes ---
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': lr_model.coef_
}).sort_values('coefficient', key=abs, ascending=True)

plt.figure(figsize=(8, 5))
colors = ['coral' if c < 0 else 'steelblue' for c in coef_df['coefficient']]
plt.barh(coef_df['feature'], coef_df['coefficient'], color=colors)
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.title('Coeficientes da Regressão Linear')
plt.xlabel('Coeficiente (impacto em log1p(Sales))')
plt.tight_layout()
plt.show()

# Observação: coeficientes positivos aumentam Sales previsto; negativos reduzem.
# Desconto (Discount) tende a ter coeficiente negativo, confirmando a análise EDA.

### Interpretação — Regressão Linear

- O modelo foi treinado com o alvo em escala logarítmica (`log1p`) para reduzir a influência dos outliers.
- O **R²** indica a proporção da variância de `Sales` explicada pelas features escolhidas.
- Um R² entre 0.3 e 0.5 já é considerado razoável neste tipo de dataset de varejo, onde fatores não observados (promoções, sazonalidade pontual) influenciam bastante o valor de cada venda.
- **Ponto de melhoria:** incluir `Sub-Category` ou o `Ship Mode` como features adicionais provavelmente elevaria o R².

---
## 4. Regressão Logística

**Objetivo:** classificar se uma venda vai gerar lucro (`Profit_flag = 1`) ou prejuízo (`Profit_flag = 0`).  
Usamos as mesmas features da regressão linear, adicionando `log1p(Sales)` como preditor.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

# --- Criação da variável alvo binária ---
df['Profit_flag'] = (df['Profit'] > 0).astype(int)

print('Distribuição da variável alvo:')
print(df['Profit_flag'].value_counts(normalize=True).map('{:.1%}'.format))

# Observação: se houver desbalanceamento acentuado (ex: 70/30),
# aplicaremos class_weight='balanced' no modelo.

In [ ]:
# --- Preparação dos dados para classificação ---
# Incluímos log1p(Sales) como feature adicional
X_log = X.copy()
X_log['log_sales'] = np.log1p(df['Sales'])

y_clf = df['Profit_flag']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_log, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Padronização (refit do scaler para incluir log_sales)
scaler_c = StandardScaler()
X_train_c = X_train_c.copy()
X_test_c  = X_test_c.copy()
num_cols_c = features_num + ['log_sales']
X_train_c[num_cols_c] = scaler_c.fit_transform(X_train_c[num_cols_c])
X_test_c[num_cols_c]  = scaler_c.transform(X_test_c[num_cols_c])

print(f'Treino: {X_train_c.shape[0]} amostras | Teste: {X_test_c.shape[0]} amostras')

In [ ]:
# --- Treinamento ---
log_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',  # compensa possível desbalanceamento
    random_state=42
)
log_model.fit(X_train_c, y_train_c)

y_pred_c      = log_model.predict(X_test_c)
y_pred_proba  = log_model.predict_proba(X_test_c)[:, 1]

# --- Métricas ---
print(f'Acurácia : {accuracy_score(y_test_c, y_pred_c):.4f}')
print(f'AUC-ROC  : {roc_auc_score(y_test_c, y_pred_proba):.4f}')
print('\nRelatório de classificação:')
print(classification_report(y_test_c, y_pred_c, target_names=['Prejuízo', 'Lucro']))

In [ ]:
# --- Matriz de confusão ---
fig, axes = plt.subplots(1, 2)

cm = confusion_matrix(y_test_c, y_pred_c)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Prejuízo', 'Lucro'],
    yticklabels=['Prejuízo', 'Lucro'],
    ax=axes[0]
)
axes[0].set_title('Matriz de Confusão')
axes[0].set_xlabel('Previsto')
axes[0].set_ylabel('Real')

# Curva ROC
fpr, tpr, _ = roc_curve(y_test_c, y_pred_proba)
auc = roc_auc_score(y_test_c, y_pred_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatório')
axes[1].set_title('Curva ROC')
axes[1].set_xlabel('Taxa de Falso Positivo')
axes[1].set_ylabel('Taxa de Verdadeiro Positivo')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Odds Ratio: interpretação dos coeficientes ---
odds_df = pd.DataFrame({
    'feature'   : X_train_c.columns,
    'odds_ratio': np.exp(log_model.coef_[0])
}).sort_values('odds_ratio', ascending=True)

plt.figure(figsize=(8, 5))
colors = ['coral' if v < 1 else 'steelblue' for v in odds_df['odds_ratio']]
plt.barh(odds_df['feature'], odds_df['odds_ratio'], color=colors)
plt.axvline(1, color='black', linestyle='--', linewidth=0.8, label='sem efeito')
plt.title('Odds Ratio — Regressão Logística')
plt.xlabel('Odds Ratio (exp do coeficiente)')
plt.legend()
plt.tight_layout()
plt.show()

# Interpretação: OR > 1 aumenta a chance de lucro; OR < 1 reduz.
# Ex: Discount com OR < 1 confirma que descontos altos aumentam o risco de prejuízo.

### Interpretação — Regressão Logística

- O **AUC-ROC** mede a capacidade do modelo de distinguir vendas com lucro das com prejuízo. Um valor acima de 0.70 já indica discriminação útil para tomada de decisão.
- O **Odds Ratio** é a forma mais intuitiva de interpretar os coeficientes: um OR de 0.3 para `Discount` significa que, ao aumentar o desconto em 1 desvio-padrão, a chance de lucro cai para 30% do valor original.
- **Ponto de melhoria:** testar um limiar de decisão diferente de 0.5 (ex: 0.4) pode melhorar o recall para a classe `Prejuízo`, que é a de maior interesse para o negócio.

---
## 5. Séries Temporais — Prophet

**Objetivo:** modelar a receita diária da loja e prever os próximos 90 dias.  
O Prophet espera um DataFrame com colunas `ds` (data) e `y` (valor).

In [ ]:
from prophet import Prophet

# --- Criação da série temporal diária ---
# Agrupamos as vendas por dia de pedido
ts = (
    df.groupby('Order Date')['Sales']
    .sum()
    .reset_index()
    .rename(columns={'Order Date': 'ds', 'Sales': 'y'})
)

# Reindexamos para garantir continuidade — dias sem vendas recebem 0
full_range = pd.date_range(ts['ds'].min(), ts['ds'].max(), freq='D')
ts = ts.set_index('ds').reindex(full_range, fill_value=0).reset_index()
ts.columns = ['ds', 'y']

print(f'Série temporal: {len(ts)} dias | {ts["ds"].min().date()} → {ts["ds"].max().date()}')
ts.tail()

In [ ]:
# --- Visualização da série bruta ---
plt.figure(figsize=(14, 4))
plt.plot(ts['ds'], ts['y'], linewidth=0.6, color='steelblue', alpha=0.8)
plt.title('Receita diária — Superstore (2015–2018)')
plt.xlabel('Data')
plt.ylabel('Sales (USD)')
plt.tight_layout()
plt.show()

# Observação: é possível observar picos no final de cada ano (Black Friday / Natal)
# e uma tendência de crescimento ao longo do período.

In [ ]:
# --- Treinamento do modelo Prophet ---
model = Prophet(
    yearly_seasonality=True,   # captura sazonalidade anual (ex: picos de fim de ano)
    weekly_seasonality=True,   # captura padrão dia da semana
    daily_seasonality=False,   # não há variação intradiária neste dataset
    changepoint_prior_scale=0.05  # regularização: evita overfitting em tendência
)

model.fit(ts)
print('Modelo treinado com sucesso.')

In [ ]:
# --- Previsão para os próximos 90 dias ---
future = model.make_future_dataframe(periods=90, freq='D')
forecast = model.predict(future)

# As colunas principais do forecast:
# yhat        → previsão central
# yhat_lower  → limite inferior do intervalo de incerteza
# yhat_upper  → limite superior do intervalo de incerteza
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10)

In [ ]:
# --- Gráfico da previsão ---
fig1 = model.plot(forecast, figsize=(14, 5))
plt.title('Previsão de vendas diárias — Prophet (+90 dias)')
plt.xlabel('Data')
plt.ylabel('Sales (USD)')
plt.tight_layout()
plt.show()

# Pontos pretos = dados reais | Linha azul = previsão | Área azul = intervalo de incerteza

In [ ]:
# --- Decomposição dos componentes ---
# O Prophet decompõe a série em: tendência + sazonalidade semanal + sazonalidade anual
fig2 = model.plot_components(forecast, figsize=(12, 8))
plt.tight_layout()
plt.show()

# Observação: o componente semanal revela quais dias da semana têm mais vendas;
# o componente anual mostra os meses de pico — informação valiosa para planejamento.

In [ ]:
# --- Avaliação: MAE e RMSE nos dados históricos ---
# Comparamos previsão in-sample com os valores reais
merged = ts.merge(forecast[['ds', 'yhat']], on='ds')

mae_ts  = mean_absolute_error(merged['y'], merged['yhat'])
rmse_ts = np.sqrt(mean_squared_error(merged['y'], merged['yhat']))

print(f'MAE  (in-sample): R$ {mae_ts:,.2f}')
print(f'RMSE (in-sample): R$ {rmse_ts:,.2f}')

# Observação: erro in-sample é otimista (modelo viu os dados de treino).
# Para uma avaliação mais rigorosa, seria necessário cross-validation temporal
# com a função prophet.diagnostics.cross_validation().

---
## 6. Conclusão

### Resumo dos resultados

| Modelo | Métrica principal | Resultado |
|--------|------------------|-----------|
| Regressão Linear | R² (log-Sales) | ver saída acima |
| Regressão Logística | AUC-ROC | ver saída acima |
| Série Temporal (Prophet) | MAE in-sample | ver saída acima |

### Aprendizados

1. **Transformações importam:** o uso de `log1p` na regressão linear foi essencial para lidar com a assimetria de `Sales` e melhorar o ajuste do modelo.

2. **Desconto como sinal de risco:** tanto na regressão linear quanto na logística, `Discount` foi um dos preditores mais relevantes — confirmando que políticas de desconto mal calibradas corroem a margem.

3. **Prophet é intuitivo para negócios:** a decomposição em componentes (tendência + sazonalidade) gera insights diretamente acionáveis, como os dias da semana e meses com maior receita esperada.

### Pontos de melhoria

- **Regressão Linear:** incluir `Sub-Category` como feature e testar modelos regularizados (Ridge, Lasso) para reduzir overfitting em categorias com poucos dados.
- **Regressão Logística:** explorar outros limiares de decisão além de 0.5 para priorizar a detecção de vendas com prejuízo (maior recall para classe 0).
- **Séries Temporais:** usar `cross_validation()` do Prophet para uma estimativa de erro mais confiável; adicionar feriados americanos como regressores externos.
- **Integração dos modelos:** um pipeline que usa a previsão de Sales do Prophet como feature para a regressão logística poderia antecipar vendas de baixa margem antes mesmo de os pedidos serem realizados.